In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import tensorflow as tf

from func_file_Model import ResNet_model_paper
from func_file_Double_star import *

In [ ]:
#Low-resolution image star coordinates [px] and uncertainties [px]
star_1_lr_x, star_1_lr_y, star_2_lr_x, star_2_lr_y = Get_table_information()[0]
star_1_lr_x_unc, star_1_lr_y_unc, star_2_lr_x_unc, star_2_lr_y_unc = Get_table_information()[1]

#Upsampled high-resolution image star coordinates [up_px]
star_1_hr_x, star_1_hr_y = Upsample_coordinates(star_1_lr_x, star_1_lr_y)
star_2_hr_x, star_2_hr_y = Upsample_coordinates(star_2_lr_x, star_2_lr_y)

#Emitted star intensities
star_1_int, star_2_int = Get_table_information()[-2]

### Load model

Note: During loading, tensorflow warning may be printed

In [ ]:
#Load the model
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_ResNet.keras")

### Reconstruct binary star image

In [ ]:
#Load and process the image
image = Load_image()

predicted = model_ResNet.predict(Normalize_data(image[None,:,:])[:,:,:,None]).squeeze()
predicted.shape

In [ ]:
#Visualize LR image
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.matshow(image, cmap="viridis", fignum=False)
plt.title("Low-resolution image")

#and its HR upsampled reconstruction
plt.subplot(122)
plt.matshow(predicted, cmap="viridis", fignum=False, vmax=0.35*predicted.max())
plt.plot(star_1_hr_x, star_1_hr_y, '+', color="red")
plt.plot(star_2_hr_x, star_2_hr_y, '+', color="red")
plt.title("High-resolution reconstruction")

plt.show()

## Zoom to the relevant region for visualization

In [ ]:
#Specify the zoom area
zoom_vt = 209
zoom_vb = 249
zoom_hl = 205
zoom_hr = 245

predicted_zoom = predicted[zoom_vt:zoom_vb, zoom_hl:zoom_hr]
star_1_hr_x_zoomed, star_1_hr_y_zoomed = Zoom_coordinates(star_1_hr_x, star_1_hr_y, zoom_hl, zoom_vt)
star_2_hr_x_zoomed, star_2_hr_y_zoomed = Zoom_coordinates(star_2_hr_x, star_2_hr_y, zoom_hl, zoom_vt)

In [ ]:
#Keep the LR image
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.matshow(image, cmap="viridis", fignum=False)
plt.title("Low-resolution image")

#and zoom on the binary star
plt.subplot(122)
plt.matshow(predicted_zoom, fignum=False)
plt.plot(star_1_hr_x_zoomed, star_1_hr_y_zoomed, '+', color="red", markersize=12)
plt.plot(star_2_hr_x_zoomed, star_2_hr_y_zoomed, '+', color="red", markersize=12)
plt.title("High-resolution zoom")

In [ ]:
#Keep the LR image
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.matshow(image, cmap="viridis", fignum=False)
plt.title("Low-resolution image")

#Add uncertainity to the high-resolution image
plt.subplot(122)
plt.matshow(predicted_zoom, fignum=False)
plt.plot(star_1_hr_x_zoomed, star_1_hr_y_zoomed, '+', color="red", markersize=12)
plt.errorbar(star_1_hr_x_zoomed, star_1_hr_y_zoomed, xerr=8*star_1_lr_x_unc, yerr=8*star_1_lr_y_unc, 
             fmt='+', capsize=6, capthick=1, elinewidth=1, markersize=6, color="red")
plt.errorbar(star_2_hr_x_zoomed, star_2_hr_y_zoomed, xerr=8*star_2_lr_x_unc, yerr=8*star_2_lr_y_unc, 
             fmt='+', capsize=6, capthick=1, elinewidth=1, markersize=6, color="red")
plt.title("High-resolution zoom")

plt.show()

### Calculate separation

In [ ]:
#The stars separation according to the ground truth table
table_sep_arcsec, table_sep_pixels = Get_table_information()[2]
print("Separation between the stars is", np.round(table_sep_arcsec,4), "arcsec read directly from the table")
print("Separation between the stars is", np.round(table_sep_pixels,4), "pixels of the original grid, calculated from table star coordinates\n")

pixel_size = table_sep_arcsec / table_sep_pixels   #arcsec pix⁻¹
print("Approximate pixel size", np.round(pixel_size,4), "arcsec pix⁻¹")

In [ ]:
#Estimate the center of mass of each star in the reconstructed image
radius_around_max = 3

#Estimate the COM around the first peak
upper_star_max_ver, upper_star_max_hor = np.unravel_index(np.argmax(predicted_zoom[:20,:20]), predicted_zoom[:20,:20].shape)
upper_star_COM_ver, upper_star_COM_hor = scipy.ndimage.center_of_mass(predicted_zoom[upper_star_max_ver-radius_around_max:upper_star_max_ver+radius_around_max+1, 
                                                                      upper_star_max_hor-radius_around_max:upper_star_max_hor+radius_around_max+1])
upper_star_COM_ver += upper_star_max_ver - radius_around_max
upper_star_COM_hor += upper_star_max_hor - radius_around_max
print("Estimated center of mass of the upper star is at the [" + str(np.round(upper_star_COM_ver,2)) + ", " + str(np.round(upper_star_COM_hor,2)) + "] pixel position in the zoomed reconstruction")

#Estimate the COM around the second peak
lower_star_max_ver, lower_star_max_hor = np.array((20,20)) + np.unravel_index(np.argmax(predicted_zoom[20:,20:]), predicted_zoom[20:,20:].shape)
lower_star_COM_ver, lower_star_COM_hor = scipy.ndimage.center_of_mass(predicted_zoom[lower_star_max_ver-radius_around_max:lower_star_max_ver+radius_around_max+1, 
                                                                      lower_star_max_hor-radius_around_max:lower_star_max_hor+radius_around_max+1])
lower_star_COM_ver += lower_star_max_ver - radius_around_max
lower_star_COM_hor += lower_star_max_hor - radius_around_max
print("Estimated center of mass of the lower star is at the [" + str(np.round(lower_star_COM_ver,2)) + ", " + str(np.round(lower_star_COM_hor,2)) + "] pixel position in the zoomed reconstruction")

In [ ]:
#Estimate the separation between the reconstructed stars in the non-upsampled grid
estimated_sep_pixels = 1/8 * np.sqrt((upper_star_COM_ver - lower_star_COM_ver)**2 + (upper_star_COM_hor - lower_star_COM_hor)**2)
print("Estimated separation between the stars is", np.round(estimated_sep_pixels,4), "pixels of the original grid, calculated from reconstruction image")

estimated_sep_arcsec = estimated_sep_pixels * pixel_size
print("Estimated separation between the stars is", np.round(estimated_sep_arcsec,4), "arcsec")

In [ ]:
#Compare the relative intensities
star_2_int_est, star_1_int_est = np.sort(np.ndarray.flatten(predicted_zoom))[-2:]
print("Estimated relative intensities between the stars is", np.round(star_1_int_est / star_2_int_est, 2))

print("Expected relative intensities between the stars is", np.round(star_1_int / star_2_int, 2))